<a href="https://colab.research.google.com/github/AyeshaS-13/CSA4705-DEEP-LEARNING/blob/main/UNIT_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

49.To develop an RNN model that classifies product reviews as positive or negative.

In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

vocab_size = 10000
max_length = 200

(x_train, y_train), (x_test, y_test) = imdb.load_data(
    num_words=vocab_size
)

x_train = pad_sequences(x_train, maxlen=max_length)
x_test = pad_sequences(x_test, maxlen=max_length)

model = Sequential([
    Embedding(vocab_size, 64),
    SimpleRNN(64),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    x_train, y_train,
    epochs=3,
    batch_size=64,
    validation_split=0.2
)

loss, accuracy = model.evaluate(x_test, y_test)

print("Test Accuracy:", accuracy)
print("Test Loss:", loss)

50.To implement a character-level RNN that learns character patterns and generates new text.

In [ ]:
import tensorflow as tf
import numpy as np

text = open("input.txt", encoding="utf-8").read()

chars = sorted(set(text))
char_to_id = {c: i for i, c in enumerate(chars)}
id_to_char = np.array(chars)

data = np.array([char_to_id[c] for c in text])

seq_length = 40

x = []
y = []

for i in range(len(data) - seq_length):
    x.append(data[i:i + seq_length])
    y.append(data[i + seq_length])

x = np.array(x)
y = np.array(y)

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(len(chars), 64),
    tf.keras.layers.SimpleRNN(128),
    tf.keras.layers.Dense(len(chars))
])

model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    )
)

model.fit(x, y, epochs=10, batch_size=64)

seed = text[:seq_length]
generated = seed

for _ in range(300):
    ids = [char_to_id[c] for c in generated[-seq_length:]]
    prediction = model.predict(
        np.array([ids]), verbose=0
    )

    next_id = np.argmax(prediction[0])
    generated += id_to_char[next_id]

print("Generated Text:\n")
print(generated)

51. RNN-Based Stock Price Forecasting

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

try:
    data = pd.read_csv("stock.csv")["Close"].values
except FileNotFoundError:
    np.random.seed(42)
    data = 100 + np.cumsum(np.random.randn(500))

scaler = MinMaxScaler()
scaled = scaler.fit_transform(data.reshape(-1, 1))

X, y = [], []

for i in range(20, len(scaled)):
    X.append(scaled[i-20:i])
    y.append(scaled[i])

X = np.array(X)
y = np.array(y)

split = int(len(X) * 0.8)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

model = tf.keras.Sequential([
    tf.keras.layers.SimpleRNN(
        50, activation="tanh",
        input_shape=(20, 1)
    ),
    tf.keras.layers.Dense(1)
])

model.compile(optimizer="adam", loss="mse")

model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    verbose=1
)

pred = model.predict(X_test)

actual = scaler.inverse_transform(y_test)
predicted = scaler.inverse_transform(pred)

print("MAE:", mean_absolute_error(actual, predicted))
print("MSE:", mean_squared_error(actual, predicted))
print("RMSE:", np.sqrt(mean_squared_error(actual, predicted)))

52. RNN-Based Electricity Consumption Forecasting

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error

np.random.seed(42)

days = np.arange(500)
consumption = (
    100 + 15 * np.sin(2 * np.pi * days / 7)
    + 0.05 * days
    + np.random.normal(0, 3, 500)
)

scaler = MinMaxScaler()
data = scaler.fit_transform(consumption.reshape(-1, 1))

X, y = [], []

for i in range(14, len(data)):
    X.append(data[i-14:i])
    y.append(data[i])

X = np.array(X)
y = np.array(y)

split = int(len(X) * 0.8)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

model = tf.keras.Sequential([
    tf.keras.layers.SimpleRNN(
        64, activation="tanh",
        input_shape=(14, 1)
    ),
    tf.keras.layers.Dense(1)
])

model.compile(optimizer="adam", loss="mse")

model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32
)

pred = model.predict(X_test)

actual = scaler.inverse_transform(y_test)
predicted = scaler.inverse_transform(pred)

print("Actual Consumption:", actual[:5].flatten())
print("Predicted Consumption:", predicted[:5].flatten())
print("MAE:", mean_absolute_error(actual, predicted))

53. RNN-Based Next-Word Prediction

In [ ]:
import tensorflow as tf
import numpy as np

text = """
deep learning is useful for prediction
deep learning is used in image processing
recurrent neural networks process sequential data
neural networks learn patterns from data
"""

tokenizer = tf.keras.preprocessing.text.Tokenizer()
tokenizer.fit_on_texts([text])

sequences = []

for line in text.strip().split("\n"):
    tokens = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(tokens)):
        sequences.append(tokens[:i+1])

max_len = max(len(s) for s in sequences)

sequences = tf.keras.preprocessing.sequence.pad_sequences(
    sequences, maxlen=max_len
)

X = sequences[:, :-1]
y = sequences[:, -1]

vocab_size = len(tokenizer.word_index) + 1

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, 32),
    tf.keras.layers.SimpleRNN(64),
    tf.keras.layers.Dense(vocab_size, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(X, y, epochs=100, verbose=0)

sentence = "deep learning"
tokens = tokenizer.texts_to_sequences([sentence])[0]

tokens = tf.keras.preprocessing.sequence.pad_sequences(
    [tokens], maxlen=max_len-1
)

prediction = model.predict(tokens, verbose=0)
next_id = np.argmax(prediction)

print("Input:", sentence)
print("Next Word:", tokenizer.index_word.get(next_id, "Unknown"))

54. Comparison of RNN and LSTM for Product Demand Forecasting

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

np.random.seed(42)

sales = 200 + np.arange(500) * 0.2 + np.random.normal(0, 10, 500)

scaler = MinMaxScaler()
data = scaler.fit_transform(sales.reshape(-1, 1))

X, y = [], []

for i in range(20, len(data)):
    X.append(data[i-20:i])
    y.append(data[i])

X = np.array(X)
y = np.array(y)

split = int(len(X) * 0.8)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

def build_model(model_type):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(20, 1))
    ])

    if model_type == "RNN":
        model.add(tf.keras.layers.SimpleRNN(50))
    else:
        model.add(tf.keras.layers.LSTM(50))

    model.add(tf.keras.layers.Dense(1))

    model.compile(optimizer="adam", loss="mse")
    return model

for model_type in ["RNN", "LSTM"]:

    model = build_model(model_type)

    model.fit(
        X_train, y_train,
        epochs=20,
        batch_size=32,
        verbose=0
    )

    pred = model.predict(X_test, verbose=0)

    actual = scaler.inverse_transform(y_test)
    predicted = scaler.inverse_transform(pred)

    print("\n", model_type)
    print("MAE:", mean_absolute_error(actual, predicted))
    print("MSE:", mean_squared_error(actual, predicted))
    print("RMSE:", np.sqrt(mean_squared_error(actual, predicted)))

55. LSTM-Based Stock Price Prediction

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

try:
    prices = pd.read_csv("stock.csv")["Close"].values
except FileNotFoundError:
    np.random.seed(42)
    prices = 100 + np.cumsum(np.random.randn(500))

scaler = MinMaxScaler()
data = scaler.fit_transform(prices.reshape(-1, 1))

X, y = [], []

for i in range(30, len(data)):
    X.append(data[i-30:i])
    y.append(data[i])

X = np.array(X)
y = np.array(y)

split = int(len(X) * 0.8)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(30, 1)),
    tf.keras.layers.LSTM(64, return_sequences=True),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dense(1)
])

model.compile(optimizer="adam", loss="mse")

model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32
)

pred = model.predict(X_test)

actual = scaler.inverse_transform(y_test)
predicted = scaler.inverse_transform(pred)

print("MAE:", mean_absolute_error(actual, predicted))
print("MSE:", mean_squared_error(actual, predicted))
print("RMSE:", np.sqrt(mean_squared_error(actual, predicted)))

56. GRU-Based Multivariate Weather Forecasting

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler

np.random.seed(42)

n = 500
days = np.arange(n)

temperature = 25 + 5 * np.sin(days / 20) + np.random.randn(n)
humidity = 60 + 10 * np.sin(days / 15) + np.random.randn(n) * 2
pressure = 1010 + 5 * np.cos(days / 25) + np.random.randn(n)

data = np.column_stack([
    temperature, humidity, pressure
])

scaler = MinMaxScaler()
data = scaler.fit_transform(data)

X, y = [], []

for i in range(10, len(data)):
    X.append(data[i-10:i])
    y.append(data[i])

X = np.array(X)
y = np.array(y)

split = int(len(X) * 0.8)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(10, 3)),
    tf.keras.layers.GRU(64, return_sequences=True),
    tf.keras.layers.GRU(32),
    tf.keras.layers.Dense(3)
])

model.compile(optimizer="adam", loss="mse")

model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32
)

pred = model.predict(X_test)

actual = scaler.inverse_transform(y_test)
predicted = scaler.inverse_transform(pred)

print("Actual Weather:\n", actual[:5])
print("Predicted Weather:\n", predicted[:5])

57. Comparison of RNN, LSTM and GRU

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

np.random.seed(42)

data = (
    100 + 0.1 * np.arange(500)
    + 10 * np.sin(np.arange(500) / 15)
    + np.random.normal(0, 2, 500)
)

scaler = MinMaxScaler()
data = scaler.fit_transform(data.reshape(-1, 1))

X, y = [], []

for i in range(20, len(data)):
    X.append(data[i-20:i])
    y.append(data[i])

X = np.array(X)
y = np.array(y)

split = int(len(X) * 0.8)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

def create_model(kind):

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(20, 1))
    ])

    if kind == "RNN":
        model.add(tf.keras.layers.SimpleRNN(50))

    elif kind == "LSTM":
        model.add(tf.keras.layers.LSTM(50))

    else:
        model.add(tf.keras.layers.GRU(50))

    model.add(tf.keras.layers.Dense(1))

    model.compile(optimizer="adam", loss="mse")

    return model

for kind in ["RNN", "LSTM", "GRU"]:

    model = create_model(kind)

    model.fit(
        X_train, y_train,
        epochs=20,
        batch_size=32,
        verbose=0
    )

    pred = model.predict(X_test, verbose=0)

    actual = scaler.inverse_transform(y_test)
    predicted = scaler.inverse_transform(pred)

    print("\nArchitecture:", kind)
    print("MAE:", mean_absolute_error(actual, predicted))
    print("MSE:", mean_squared_error(actual, predicted))
    print("RMSE:", np.sqrt(mean_squared_error(actual, predicted)))

58. Stacked RNN for Machine Condition Prediction

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler

np.random.seed(42)

n = 500
temperature = 50 + np.random.randn(n) * 3
pressure = 100 + np.random.randn(n) * 5
vibration = 10 + np.random.randn(n) * 2

data = np.column_stack([
    temperature, pressure, vibration
])

scaler = MinMaxScaler()
data = scaler.fit_transform(data)

X, y = [], []

for i in range(15, len(data)):
    X.append(data[i-15:i])
    y.append(data[i])

X = np.array(X)
y = np.array(y)

split = int(len(X) * 0.8)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(15, 3)),
    tf.keras.layers.SimpleRNN(64, return_sequences=True),
    tf.keras.layers.SimpleRNN(32),
    tf.keras.layers.Dense(3)
])

model.compile(optimizer="adam", loss="mse")

model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32
)

pred = model.predict(X_test)

actual = scaler.inverse_transform(y_test)
predicted = scaler.inverse_transform(pred)

print("Actual Readings:\n", actual[:5])
print("Predicted Readings:\n", predicted[:5])

59. Demonstration of the Vanishing Gradient Problem

In [ ]:
import torch

torch.manual_seed(42)

weights = [0.5, 0.5, 0.5, 0.5, 0.5,
           0.5, 0.5, 0.5, 0.5, 0.5]

print("Gradient Values Across Time Steps")

for steps in range(1, 11):

    gradient = torch.tensor(1.0)

    for i in range(steps):
        gradient = gradient * weights[i]

    print(
        "Time Steps:", steps,
        "Gradient:", gradient.item()
    )

60. Effect of Sequence Length on RNN Forecasting

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

np.random.seed(42)

data = (
    25 + 5 * np.sin(np.arange(500) / 20)
    + np.random.normal(0, 1, 500)
)

scaler = MinMaxScaler()
data = scaler.fit_transform(data.reshape(-1, 1))

for seq_length in [5, 10, 20, 30]:

    X, y = [], []

    for i in range(seq_length, len(data)):
        X.append(data[i-seq_length:i])
        y.append(data[i])

    X = np.array(X)
    y = np.array(y)

    split = int(len(X) * 0.8)

    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(seq_length, 1)),
        tf.keras.layers.SimpleRNN(32),
        tf.keras.layers.Dense(1)
    ])

    model.compile(optimizer="adam", loss="mse")

    model.fit(
        X_train, y_train,
        epochs=10,
        batch_size=32,
        verbose=0
    )

    pred = model.predict(X_test, verbose=0)

    error = np.sqrt(mean_squared_error(y_test, pred))

    print(
        "Sequence Length:", seq_length,
        "RMSE:", error
    )

61. LSTM Hyperparameter Tuning

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler

np.random.seed(42)

data = (
    100 + 0.2 * np.arange(400)
    + 10 * np.sin(np.arange(400) / 15)
    + np.random.normal(0, 2, 400)
)

scaler = MinMaxScaler()
data = scaler.fit_transform(data.reshape(-1, 1))

X, y = [], []

for i in range(20, len(data)):
    X.append(data[i-20:i])
    y.append(data[i])

X = np.array(X)
y = np.array(y)

split = int(len(X) * 0.8)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

best_loss = float("inf")
best_config = None

for lr in [0.001, 0.01]:
    for units in [32, 64]:
        for batch in [16, 32]:

            model = tf.keras.Sequential([
                tf.keras.layers.Input(shape=(20, 1)),
                tf.keras.layers.LSTM(units),
                tf.keras.layers.Dense(1)
            ])

            model.compile(
                optimizer=tf.keras.optimizers.Adam(
                    learning_rate=lr
                ),
                loss="mse"
            )

            history = model.fit(
                X_train, y_train,
                validation_data=(X_test, y_test),
                epochs=10,
                batch_size=batch,
                verbose=0
            )

            loss = min(history.history["val_loss"])

            print(
                "LR:", lr,
                "Units:", units,
                "Batch:", batch,
                "Validation Loss:", loss
            )

            if loss < best_loss:
                best_loss = loss
                best_config = (lr, units, batch)

print("\nBest Configuration:", best_config)
print("Best Validation Loss:", best_loss)

62. Evaluation of RNN Forecasting Errors

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

np.random.seed(42)

data = (
    100 + 10 * np.sin(np.arange(400) / 10)
    + np.random.normal(0, 2, 400)
)

X, y = [], []

for i in range(10, len(data)):
    X.append(data[i-10:i])
    y.append(data[i])

X = np.array(X)
y = np.array(y)

X = X.reshape(X.shape[0], X.shape[1], 1)

split = int(len(X) * 0.8)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(10, 1)),
    tf.keras.layers.SimpleRNN(32),
    tf.keras.layers.Dense(1)
])

model.compile(optimizer="adam", loss="mse")

model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    verbose=0
)

pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
mse = mean_squared_error(y_test, pred)
rmse = np.sqrt(mse)

print("Mean Absolute Error:", mae)
print("Mean Squared Error:", mse)
print("Root Mean Squared Error:", rmse)

63. RNN-Based Customer Complaint Classification

In [ ]:
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

texts = [
    "My payment was deducted twice",
    "I cannot complete my online payment",
    "The payment failed",
    "My order has not arrived",
    "The delivery is delayed",
    "I received the wrong product",
    "The product is damaged",
    "The item is defective",
    "I want to return my product",
    "I need a refund",
    "Please cancel my order",
    "My refund is not processed",
    "The payment was successful",
    "Where is my delivery",
    "The product quality is poor",
    "I received a broken item",
    "My money was not refunded",
    "The order is still pending"
]

labels = [
    0, 0, 0,
    1, 1, 1,
    2, 2, 2,
    3, 3, 3,
    0, 1, 2, 2, 3, 1
]

X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.25,
    random_state=42,
    stratify=labels
)

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=1000,
    output_sequence_length=20
)

vectorizer.adapt(X_train)

model = tf.keras.Sequential([
    vectorizer,
    tf.keras.layers.Embedding(1000, 32),
    tf.keras.layers.SimpleRNN(32),
    tf.keras.layers.Dense(4, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    np.array(X_train),
    np.array(y_train),
    epochs=30,
    batch_size=4,
    verbose=0
)

pred = model.predict(
    np.array(X_test),
    verbose=0
)

y_pred = np.argmax(pred, axis=1)

print("Classification Report:")

print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1, 2, 3],
    target_names=[
        "Payment",
        "Delivery",
        "Product",
        "Refund"
    ],
    zero_division=0
))

64. Recursive Neural Network for Sentiment Classification

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(42)

# Word embeddings
words = [
    "good", "excellent", "bad", "terrible",
    "very", "not"
]

word_to_id = {w: i for i, w in enumerate(words)}

# Recursive Neural Network
class RecursiveNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(len(words), 16)

        self.compose = nn.Linear(32, 16)

        self.output = nn.Linear(16, 2)

    def combine(self, left, right):

        x = torch.cat((left, right), dim=1)

        return torch.tanh(self.compose(x))

    def leaf(self, word):

        index = torch.tensor([word_to_id[word]])

        return self.embedding(index)

    def forward(self, tree):

        if isinstance(tree, str):
            return self.leaf(tree)

        left = self.forward(tree[0])
        right = self.forward(tree[1])

        return self.combine(left, right)


model = RecursiveNN()

optimizer = optim.Adam(model.parameters(), lr=0.01)

criterion = nn.CrossEntropyLoss()

# Training examples: binary trees and sentiment labels
data = [
    (("very", "good"), 1),
    (("not", "good"), 0),
    (("very", "excellent"), 1),
    (("not", "bad"), 1),
    (("very", "bad"), 0),
    (("not", "terrible"), 1)
]

# Training
for epoch in range(100):

    total_loss = 0

    for tree, label in data:

        optimizer.zero_grad()

        representation = model(tree)

        prediction = model.output(representation)

        target = torch.tensor([label])

        loss = criterion(prediction, target)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 20 == 0:
        print(
            "Epoch:", epoch + 1,
            "Loss:", total_loss
        )


# Testing
test_sentences = [
    (("very", "good"), "Positive"),
    (("not", "good"), "Negative"),
    (("very", "bad"), "Negative")
]

print("\nSentiment Predictions:")

for tree, sentence in test_sentences:

    with torch.no_grad():

        representation = model(tree)

        prediction = model.output(representation)

        result = torch.argmax(
            prediction, dim=1
        ).item()

    sentiment = "Positive" if result == 1 else "Negative"

    print(sentence, ":", sentiment)